In [2]:
// 1: Declare all dependencies (evcxr Jupyter syntax)
:dep reqwest = { version = "0.11", default-features = false, features = ["blocking", "rustls-tls", "json"] }
:dep polars = { version = "0.36", features = ["lazy", "temporal", "strings", "parquet"] }
:dep serde_json = "1.0"
:dep anyhow = "1.0"

use reqwest::blocking::{Client, Response};
use polars::prelude::*;
use serde_json::Value;
use anyhow::{Result, Context, anyhow};
use std::time::Duration;
use std::thread;
use std::fs::File;

// 2: Run the historical data fetch with Error Handling & Explicit Types
fn fetch_historical_data() -> Result<DataFrame> {
    // Explicit type declaration for the HTTP client
    let client: Client = Client::new();
    
    // Explicit type declaration for our MVP coins array
    let mvp_coins: Vec<&str> = vec!["bitcoin", "ethereum", "binancecoin", "solana"];
    let days_to_fetch: i32 = 365;
    
    // Vector to hold individual DataFrames before concatenation
    let mut dataframes: Vec<DataFrame> = Vec::new();

    for coin_id in mvp_coins {
        println!("Fetching 365-day history for: {}", coin_id);
        
        let url: String = format!(
            "https://api.coingecko.com/api/v3/coins/{}/market_chart?vs_currency=usd&days={}", 
            coin_id, days_to_fetch
        );

        // --- ERROR HANDLING 1: HTTP Request Failure ---
        // Using .context() to add readable error messages if the request fails
        let response: Response = client.get(&url)
            .header("User-Agent", "RustDataPlatform/1.0")
            .send()
            .context(format!("Failed to send HTTP request to CoinGecko for {}", coin_id))?;

        // --- ERROR HANDLING 2: Non-200 Status Code ---
        if !response.status().is_success() {
            return Err(anyhow!("API Error for {}. Status Code: {}", coin_id, response.status()));
        }

        // Parse JSON
        let json_data: Value = response.json()
            .context("Failed to deserialize response body into JSON")?;

        // --- ERROR HANDLING 3: Missing JSON Keys ---
        let prices_array: &Vec<Value> = json_data["prices"].as_array()
            .context(format!("Missing or invalid 'prices' array for {}", coin_id))?;
        let market_caps_array: &Vec<Value> = json_data["market_caps"].as_array()
            .context(format!("Missing or invalid 'market_caps' array for {}", coin_id))?;
        let volumes_array: &Vec<Value> = json_data["total_volumes"].as_array()
            .context(format!("Missing or invalid 'total_volumes' array for {}", coin_id))?;

        let row_count: usize = prices_array.len();

        // Initialize typed vectors to build the Polars Series
        let mut timestamps: Vec<i64> = Vec::with_capacity(row_count);
        let mut prices: Vec<f64> = Vec::with_capacity(row_count);
        let mut market_caps: Vec<f64> = Vec::with_capacity(row_count);
        let mut volumes: Vec<f64> = Vec::with_capacity(row_count);
        let coin_ids: Vec<&str> = vec![coin_id; row_count];

        // Flatten the nested JSON arrays
        for i in 0..row_count {
            // Using .unwrap_or() for safe fallback in case of weird API nulls, 
            // though DQ checks will catch 0.0 later.
            let ts: i64 = prices_array[i][0].as_i64().unwrap_or(0);
            let px: f64 = prices_array[i][1].as_f64().unwrap_or(0.0);
            
            // Safely fetch corresponding rows from caps and volumes
            let mc: f64 = market_caps_array.get(i).and_then(|v| v[1].as_f64()).unwrap_or(0.0);
            let vol: f64 = volumes_array.get(i).and_then(|v| v[1].as_f64()).unwrap_or(0.0);

            // --- ERROR HANDLING 4: Data Quality Checks ---
            if px <= 0.0 || ts == 0 {
                return Err(anyhow!("Data Quality Violation: Invalid price or timestamp found for {} at index {}", coin_id, i));
            }

            timestamps.push(ts);
            prices.push(px);
            market_caps.push(mc);
            volumes.push(vol);
        }

        // Construct Typed Polars Series
        let s_coin_id: Series = Series::new("coin_id", coin_ids);
        let s_timestamp: Series = Series::new("timestamp_ms", timestamps);
        let s_price: Series = Series::new("price_usd", prices);
        let s_market_cap: Series = Series::new("market_cap_usd", market_caps);
        let s_volume: Series = Series::new("volume_24h_usd", volumes);

        // Build the DataFrame for this specific coin
        let df: DataFrame = DataFrame::new(vec![
            s_coin_id, s_timestamp, s_price, s_market_cap, s_volume
        ]).context(format!("Failed to construct Polars DataFrame for {}", coin_id))?;

        dataframes.push(df);

        // Sleep to respect the Free Tier Rate Limits (approx 10-30 calls/min)
        thread::sleep(Duration::from_secs(3));
    }

    // --- ERROR HANDLING 5: Empty Data ---
    if dataframes.is_empty() {
        return Err(anyhow!("No data was fetched, dataframe list is empty."));
    }

    // Vertically concatenate all DataFrames into a single Unified Table
    let mut final_df: DataFrame = dataframes[0].clone();
    for i in 1..dataframes.len() {
        final_df = final_df.vstack(&dataframes[i])
            .context("Failed to concatenate DataFrames during vertical stack")?;
    }

    Ok(final_df)
}

match fetch_historical_data() {
    Ok(mut df) => { // <-- Note: 'df' must be mutable for the writer
        println!("Successfully fetched and merged historical data!");
        println!("{:?}", df.head(Some(5))); 
        
        // --- NEW: Write the DataFrame to Parquet ---
        let file_name = "historical_rust_output.parquet";
        
        // 1. Create the physical file on your SSD
        let mut file = File::create(file_name)
            .expect("Failed to create the parquet file on disk");
            
        // 2. Initialize the Polars ParquetWriter and compress it
        ParquetWriter::new(&mut file)
            .with_compression(ParquetCompression::Snappy) // Standard, fast compression
            .finish(&mut df)
            .expect("Failed to encode DataFrame into Parquet format");
            
        println!("✅ Data successfully written to: {}", file_name);
    },
    Err(e) => {
        eprintln!("❌ Pipeline Failed with Error:\n{}", e);
    }
}

Fetching 365-day history for: bitcoin
Fetching 365-day history for: ethereum
Fetching 365-day history for: binancecoin
Fetching 365-day history for: solana
Successfully fetched and merged historical data!
shape: (5, 5)
┌─────────┬───────────────┬───────────────┬────────────────┬────────────────┐
│ coin_id ┆ timestamp_ms  ┆ price_usd     ┆ market_cap_usd ┆ volume_24h_usd │
│ ---     ┆ ---           ┆ ---           ┆ ---            ┆ ---            │
│ str     ┆ i64           ┆ f64           ┆ f64            ┆ f64            │
╞═════════╪═══════════════╪═══════════════╪════════════════╪════════════════╡
│ bitcoin ┆ 1748649600000 ┆ 104010.919562 ┆ 2.0668e12      ┆ 3.9680e10      │
│ bitcoin ┆ 1748736000000 ┆ 104687.507429 ┆ 2.0805e12      ┆ 2.0433e10      │
│ bitcoin ┆ 1748822400000 ┆ 105710.005938 ┆ 2.1012e12      ┆ 1.7722e10      │
│ bitcoin ┆ 1748908800000 ┆ 105884.742632 ┆ 2.1042e12      ┆ 2.6314e10      │
│ bitcoin ┆ 1748995200000 ┆ 105434.477451 ┆ 2.0953e12      ┆ 2.5926e10      │
└

()